In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import dataclasses

import jax

from openpi.models import model as _model
from openpi.policies import steervla_policy
from openpi.policies import policy_config as _policy_config
from openpi.shared import download
from openpi.training import config as _config
from openpi.training import data_loader as _data_loader

In [3]:
# Load the model 
config = _config.get_config("pi05_steervla_cot_ki")
checkpoint_path = "gs://cat-logs/pi05_steervla_cot_ki/pi05_steervla_cot_ki/90000"
checkpoint_dir = download.maybe_download(checkpoint_path)

In [4]:
policy = _policy_config.create_trained_policy(config, checkpoint_dir)

In [5]:
jax.devices()

[CudaDevice(id=0), CudaDevice(id=1)]

In [6]:
example = steervla_policy.make_steervla_example()
cot_sample_kwargs = {"timing": True, "timing_per_step": True, "timing_sync": True}
policy._cot_sample_kwargs = cot_sample_kwargs
# result = policy.infer_with_cot(example)

In [7]:
import jax.numpy as jnp
import time
import numpy as np


In [8]:
inputs = jax.tree.map(lambda x: x, example)
inputs = policy._input_transform(inputs)
inputs = jax.tree.map(lambda x: jnp.asarray(x)[np.newaxis, ...], inputs)
policy._rng, rng_cot, rng_act = jax.random.split(policy._rng, 3)

sample_kwargs = dict(policy._sample_kwargs)

observation = _model.Observation.from_dict(inputs)
cot_kwargs = dict(policy._cot_sample_kwargs)

In [10]:
t_cot = time.monotonic()
cot_out = policy._sample_cot(rng_cot, observation, **cot_kwargs)
cot_ms = (time.monotonic() - t_cot) * 1000

ERROR:autoreload:Failed to reload module 'openpi.models.pi0_cot' from file '/home/carla/ogbench-carla/.venv/lib/python3.11/site-packages/openpi/models/pi0_cot.py'
Traceback (most recent call last):
  File "/home/carla/.cache/uv/archive-v0/xdMAtCCnoRJ-BtDO-l6e1/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 325, in check
    superreload(m, reload, self.old_objects)
  File "/home/carla/.cache/uv/archive-v0/xdMAtCCnoRJ-BtDO-l6e1/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 625, in superreload
    update_generic(old_obj, new_obj)
  File "/home/carla/.cache/uv/archive-v0/xdMAtCCnoRJ-BtDO-l6e1/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 451, in update_generic
    update(a, b)
  File "/home/carla/.cache/uv/archive-v0/xdMAtCCnoRJ-BtDO-l6e1/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 403, in update_class
    if update_generic(old_obj, new_obj):
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ca

[autoreload of openpi.models.pi0_cot failed: Traceback (most recent call last):
  File "/home/carla/.cache/uv/archive-v0/xdMAtCCnoRJ-BtDO-l6e1/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 325, in check
    superreload(m, reload, self.old_objects)
  File "/home/carla/.cache/uv/archive-v0/xdMAtCCnoRJ-BtDO-l6e1/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 625, in superreload
    update_generic(old_obj, new_obj)
  File "/home/carla/.cache/uv/archive-v0/xdMAtCCnoRJ-BtDO-l6e1/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 451, in update_generic
    update(a, b)
  File "/home/carla/.cache/uv/archive-v0/xdMAtCCnoRJ-BtDO-l6e1/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 403, in update_class
    if update_generic(old_obj, new_obj):
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/carla/.cache/uv/archive-v0/xdMAtCCnoRJ-BtDO-l6e1/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 45

[Pi0CoT.sample_cot timing] mr=96 ms=48 batch=1 prefix_prep_ms=4900.13 prefill_ms=3326.01


UnexpectedTracerError: Encountered an unexpected tracer. A function transformed by JAX had a side effect, allowing for a reference to an intermediate value with type float32[1152] wrapped in a DynamicJaxprTracer to escape the scope of the transformation.
JAX transformations require that functions explicitly return their outputs, and disallow saving intermediate values to global state.
The function being traced when the value leaked was create at /home/carla/ogbench-carla/.venv/lib/python3.11/site-packages/openpi/models/pi0_config.py:153 traced for jit.
------------------------------
The leaked intermediate value was created on line /home/carla/ogbench-carla/.venv/lib/python3.11/site-packages/flax/core/scope.py:968:14 (Scope.param). 
------------------------------
When the value was created, the final 5 stack frames (most recent last) excluding JAX-internal frames were:
------------------------------
/home/carla/ogbench-carla/.venv/lib/python3.11/site-packages/flax/linen/module.py:1216:

In [22]:
result

{'actions': array([[0.40796981, 0.16667454, 0.86637976, 0.49659499],
        [0.38781672, 0.17720633, 0.92886932, 0.36338228],
        [0.26975203, 0.27713298, 0.92890976, 0.36127532],
        [0.17955177, 0.34590497, 0.89277979, 0.43427665],
        [0.16410216, 0.32607069, 0.81781402, 0.56051222],
        [0.17922359, 0.28315689, 0.70921162, 0.67812468],
        [0.18382544, 0.30942294, 0.57886037, 0.79599398],
        [0.19420805, 0.37139154, 0.47835238, 0.85008112],
        [0.19198261, 0.42123973, 0.3995379 , 0.89655715],
        [0.16466924, 0.4578135 , 0.35144917, 0.9084595 ]]),
 'tokenized_reasoning': array([ 11493,    604,    476,  12485,    575,    573,   9988,   1794,
         11117,  49103,   4894, 257019, 257022,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,
             0,      0, 

In [ ]:

del model

In [ ]:
# Now load examples from the dataset

config = dataclasses.replace(config, batch_size=2)

# Create a model from the checkpoint.
model = config.model.load(_model.restore_params(checkpoint_dir / "params"))

In [ ]:
config = dataclasses.replace(config, data=dataclasses.replace(config.data, dataset_name_weight_mappings={"simlingo_dataset_all_img512_1116": 1.0}))
loader = _data_loader.create_data_loader(config, num_batches=1, skip_norm_stats=True)
obs, act = next(iter(loader))

In [ ]:
import sentencepiece
import openpi.shared.download as download
import openpi.models.tokenizer as tokenizer

tokenizer = tokenizer.CoTPaligemmaTokenizer()

sp = tokenizer._tokenizer


In [ ]:
import numpy as np
def _decode_tokens(token_ids: np.ndarray, mask: np.ndarray, tokenizer) -> str:
    """Decode a padded token array back to text using the given SP tokenizer."""
    valid = token_ids[mask.astype(bool)]
    return tokenizer.decode(valid.tolist())

In [ ]:
token = tokenizer.vocab_size - 1 - 128 - 1 
tokenizer.
_decode_tokens(np.array([token]), np.array([True]), tokenizer._tokenizer)

In [ ]:
start_subtask_text = _decode_tokens(np.array([tokenizer._start_of_subtask()]), np.array([True]), tokenizer._tokenizer)
end_subtask_text = _decode_tokens(np.array([tokenizer._end_of_subtask()]), np.array([True]), tokenizer._tokenizer)
start_reasoning_text = _decode_tokens(np.array([tokenizer._start_of_reasoning()]), np.array([True]), tokenizer._tokenizer)
end_reasoning_text = _decode_tokens(np.array([tokenizer._end_of_reasoning()]), np.array([True]), tokenizer._tokenizer)
example_prompt = "Prompt:A car is driving down the street;State:84 116;" + start_subtask_text + "Follow the road normally;" + end_subtask_text + start_reasoning_text + "The car is driving down the street;" + end_reasoning_text
print(example_prompt)
subtask = example_prompt.split(end_subtask_text)[0].split(start_subtask_text)[1]
reasoning = example_prompt.split(end_reasoning_text)[0].split(start_reasoning_text)[1]
print(subtask)
print(reasoning)
print(tokenizer._tokenizer.eos_id())

In [ ]:
string = " blahblahblah"
string.split(";")[0]


In [ ]:
prompt = _decode_tokens(obs.tokenized_prompt[0], obs.tokenized_prompt_mask[0], sp)
subtask = _decode_tokens(obs.tokenized_subtask[0], obs.tokenized_subtask_mask[0], sp)
reasoning = _decode_tokens(obs.tokenized_reasoning[0], obs.tokenized_reasoning_mask[0], sp)

In [ ]:
print("Prompt: ", prompt)
print("Subtask: ", subtask)
print("Reasoning: ", reasoning)


In [ ]:
key = jax.random.key(0)
cot = model.sample_cot(key, obs)

In [ ]:
detokenized_cot = _decode_tokens(cot["tokenized_subtask"][0], cot["tokenized_subtask_mask"][0], sp)

In [ ]:
detokenized_cot

In [ ]:

import logging

import einops
import flax.nnx as nnx
import flax.nnx.bridge as nnx_bridge
import jax
import jax.numpy as jnp
from typing_extensions import override

from openpi.models import model as _model
from openpi.models import pi0
from openpi.models import pi0_config
import openpi.models.gemma as _gemma
import openpi.models.siglip as _siglip
from openpi.shared import array_typing as at

max_subtask_len = 24


observation = _model.preprocess_observation(None, obs, train=False)
if observation.tokenized_prompt is None or observation.tokenized_prompt_mask is None:
    raise ValueError("sample_cot requires tokenized_prompt and tokenized_prompt_mask")

batch_size = observation.state.shape[0]
ms = model.max_subtask_len
mr = model.max_reasoning_len

In [ ]:
print(ms)
print(mr)

In [ ]:
# Embed images
img_tokens, img_masks, img_ar = model._embed_images(observation)
        
# Embed prompt
prompt_emb = model._embed_text_tokens(observation.tokenized_prompt)
prompt_mask = observation.tokenized_prompt_mask
        
# Construct prefix
prefix_tokens = jnp.concatenate(img_tokens + [prompt_emb], axis=1)
prefix_mask = jnp.concatenate(img_masks + [prompt_mask], axis=1)
prefix_ar = jnp.array(img_ar + [False] * prompt_emb.shape[1])
        
# Build attention mask
prefix_attn_mask = pi0.make_attn_mask(prefix_mask, prefix_ar)
positions = jnp.cumsum(prefix_mask, axis=1) - 1

# Prefill prefix
(prefix_out, _), kv_cache = model.PaliGemma.llm(
    [prefix_tokens, None], mask=prefix_attn_mask, positions=positions
)
assert prefix_out is not None

In [ ]:
prefix_tokens[0].shape

In [ ]:
rng = jax.random.key(0)
h = model._gather_last_valid_hidden(prefix_out, prefix_mask)
key_mask = prefix_mask
abs_pos = jnp.sum(prefix_mask, axis=1, keepdims=True).astype(jnp.int32)

# Buffer for subtask
sub_buf = jnp.zeros((batch_size, ms), dtype=jnp.int32)
sub_m = jnp.zeros((batch_size, ms), dtype=jnp.bool_)
rng_cur = rng

In [ ]:
print(_decode_tokens(observation.tokenized_prompt[0], observation.tokenized_prompt_mask[0], sp))

In [ ]:
print(sp.encode("|")[0])

In [ ]:
temperature = 0.9
# Generate subtask
for i in range(ms):
    print("iter: ", i)
    # Decode logits
    logits = model.PaliGemma.llm(h, method="decode_logits")[:, 0, :]
    if temperature and temperature > 0:
        # Temperature sampling
        rng_cur, step_rng = jax.random.split(rng_cur)
        tok = jax.random.categorical(step_rng, logits / jnp.maximum(temperature, 1e-6))
    else:
        # Greedy
        tok = jnp.argmax(logits, axis=-1)
    
    # Update buffer
    sub_buf = sub_buf.at[:, i].set(tok)
    sub_m = sub_m.at[:, i].set(True)
    if i == ms - 1:
        break
    
    # Embed token
    emb = model._embed_text_tokens(tok[:, None])
    key_mask = jnp.concatenate([key_mask, jnp.ones((batch_size, 1), dtype=jnp.bool_)], axis=1)
    attn_mask = key_mask[:, None, :]  # (b, 1, k_len); Module adds head axis
    (out, _), kv_cache = model.PaliGemma.llm(
        [emb, None],
        mask=attn_mask,
        positions=abs_pos,
        kv_cache=kv_cache,
    )
    assert out is not None
    h = out
    abs_pos = abs_pos + 1

In [ ]:
subtask = _decode_tokens(sub_buf[0], sub_m[0], sp)
print(subtask)

In [ ]:
rea_buf = jnp.zeros((batch_size, mr), dtype=jnp.int32)
rea_m = jnp.zeros((batch_size, mr), dtype=jnp.bool_)

for j in range(mr):
    logits = self.PaliGemma.llm(h, method="decode_logits")[:, 0, :]
    if temperature and temperature > 0:
    rng_cur, step_rng = jax.random.split(rng_cur)
    tok = jax.random.categorical(step_rng, logits / jnp.maximum(temperature, 1e-6))
else:
    tok = jnp.argmax(logits, axis=-1)
rea_buf = rea_buf.at[:, j].set(tok)
rea_m = rea_m.at[:, j].set(True)
if j == mr - 1:
    break
emb = self._embed_text_tokens(tok[:, None])
key_mask = jnp.concatenate([key_mask, jnp.ones((batch_size, 1), dtype=jnp.bool_)], axis=1)
attn_mask = key_mask[:, None, :]
(out, _), kv_cache = self.PaliGemma.llm(
    [emb, None],
    mask=attn_mask,
    positions=abs_pos,
    kv_cache=kv_cache,
)
assert out is not None
h = out
abs_pos = abs_pos + 1

In [ ]:
# Now we can sample actions and cot from the model
key = jax.random.key(0)
actions = model.sample_actions(key, obs, num_steps=10)
cot = model.sample_cot(key, obs)

In [ ]:
actions

In [ ]:
cot

In [ ]:
print("Prompt: ", _decode_tokens(obs.tokenized_prompt, obs.tokenized_prompt_mask, sp))

In [ ]:
# Now detokenize the actions and cot


In [ ]:
detokenized_cot